# WSA_04 — FSA & Horizon Comparison

**Purpose.** Compare scenario sensitivity across all six FSAs and all 24 forecast horizons.

> Run from the project repository. Outputs are generated only from the project data and frozen model artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
CONFIG = PROJECT_ROOT / "configs" / "weather_sensitivity.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/weather_sensitivity.yaml')

In [3]:
# Import modules for weather_sensitivity
from src.ontario_peak_risk.weather_sensitivity.common import load_config, ensure_dirs

In [4]:
cfg, project_root = load_config(CONFIG)
paths = ensure_dirs(cfg, project_root)
print("Project root:", project_root)

Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk


In [5]:
from src.ontario_peak_risk.weather_sensitivity.comparison import (
    summary_by_fsa_scenario,
    symmetry_table,
)

In [7]:
results = pd.read_parquet(paths["outputs_dir"] / "WSA_sensitivity_master.parquet")
summary = summary_by_fsa_scenario(results)
symmetry = symmetry_table(results)
summary.to_csv(paths["outputs_dir"] / "WSA_04_fsa_scenario_summary.csv", index=False)
symmetry.to_csv(paths["outputs_dir"] / "WSA_04_symmetry_analysis.csv", index=False)
display(summary)


,fsa,scenario,temperature_delta_c,mean_abs_forecast_delta_kwh,max_abs_forecast_delta_kwh,mean_abs_forecast_delta_pct,mean_abs_peak_risk_delta,max_abs_peak_risk_delta,alert_changes,ood_rows,caution_rows
0,L4T,+2.5C,2.5,77.294838,205.984713,0.796846,0.003478,0.032620,0,0,0
1,L4T,+5C,5.0,124.085796,293.257204,1.253187,0.017699,0.150898,3,0,0
2,L4T,-2.5C,-2.5,117.164462,267.846435,1.206130,0.004959,0.054660,0,0,0
3,L4T,-5C,-5.0,291.712413,633.727903,2.983863,0.004098,0.045094,0,0,0
4,L4T,Baseline,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0
5,M5R,+2.5C,2.5,28.673148,104.755046,0.289926,0.077564,0.386207,2,0,0
6,M5R,+5C,5.0,61.628220,158.888097,0.619037,0.120955,0.495182,3,0,0
7,M5R,-2.5C,-2.5,84.391206,222.694394,0.859796,0.048007,0.207717,3,0,0
8,M5R,-5C,-5.0,285.652038,698.611438,2.894259,0.078999,0.313939,4,0,0
9,M5R,Baseline,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0


In [8]:
display(
    symmetry.groupby(["metric", "magnitude_c"])["symmetry_residual"].agg(
        ["mean", "median", "min", "max"]
    )
)

mean     median         min         max
metric             magnitude_c                                               
forecast_delta_kwh 2.5           42.935190  22.450819 -113.061734  519.889250
                   5.0          150.890255  89.001663  -69.498724  709.248410
peak_risk_delta    2.5           -0.015189   0.000068   -0.335716    0.219820
                   5.0           -0.049855   0.000040   -0.481930    0.263272

In [9]:
horizon = (
    results.groupby(["horizon", "scenario", "temperature_delta_c"], observed=True)
    .agg(
        mean_abs_forecast_delta_kwh=("forecast_delta_kwh", lambda s: s.abs().mean()),
        mean_abs_peak_risk_delta=("peak_risk_delta", lambda s: s.abs().mean()),
        alert_changes=("alert_changed", "sum"),
    )
    .reset_index()
)
horizon.to_csv(paths["outputs_dir"] / "WSA_04_horizon_summary.csv", index=False)
print("WSA_04 RESULT: COMPLETE")

WSA_04 RESULT: COMPLETE
